# NASA SRTM Elevation Extraction

Samples NASA SRTM GL1 (30 m) elevation at each grid cell's representative query point, the same coordinate used for the NASA POWER extraction.

The elevation is used as the site `altitude` in the Ineichen-Perez clear-sky model (pvlib), where it sets the site air pressure used in the airmass correction.

Data are accessed through the OpenTopoData API (https://www.opentopodata.org), which serves the NASA SRTM GL1 v003 dataset. For points that fall on SRTM voids, SRTM 90 m (CGIAR-CSI v4.1) and then NASA/METI ASTER GDEM v3 (30 m) are used as fallbacks. The `elevation_source` column records which dataset each value came from.

In [4]:
"""
extract_srtm_elevation.py

Purpose:
    Extract NASA SRTM elevation for every retained Philippine
    spatial grid cell using the representative query latitude
    and longitude generated during grid creation.

Dataset:
    Primary:    NASA SRTM GL1 v003 (30 m)     -> OpenTopoData "srtm30m"
    Fallback 1: CGIAR-CSI SRTM v4.1 (90 m)    -> OpenTopoData "srtm90m"
    Fallback 2: NASA/METI ASTER GDEM v3 (30 m) -> OpenTopoData "aster30m"

    Fallbacks are only used for points that fall on SRTM voids
    (e.g., very small islets).

Input:
    data/processed/grid_metadata.csv

Output:
    data/raw/elevation/grid_elevation.csv
"""

from pathlib import Path
import time
import requests
import pandas as pd


# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# OPENTOPODATA DATASETS
# ------------------------------------------------------------

PRIMARY_DATASET = "srtm30m"

# Tried in order for points still missing an elevation
FALLBACK_DATASETS = [
    "srtm90m",
    "aster30m"
]


# ------------------------------------------------------------
# INPUT FILE
# ------------------------------------------------------------

GRID_FILE = Path(
    "data/processed/grid_metadata.csv"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "data/raw/elevation"
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "grid_elevation.csv"
)


# ------------------------------------------------------------
# OPENTOPODATA API
# ------------------------------------------------------------

BASE_URL = (
    "https://api.opentopodata.org/v1"
)


# ------------------------------------------------------------
# REQUEST SETTINGS
# ------------------------------------------------------------

REQUEST_TIMEOUT = 60

# Public API limit: 100 locations per request
BATCH_SIZE = 100

# Public API limit: 1 request per second
REQUEST_DELAY = 1.1


# ============================================================
# 2. CREATE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. LOAD GRID METADATA
# ============================================================

print("\nLoading grid metadata...")

if not GRID_FILE.exists():

    raise FileNotFoundError(
        f"""
Grid metadata file was not found:

{GRID_FILE}

Please run create_grid.py first.
"""
    )


grid = pd.read_csv(GRID_FILE)

print(
    f"Grid cells found: {len(grid)}"
)


# ============================================================
# 4. OPENTOPODATA API FUNCTION
# ============================================================

def get_elevations(
    latitudes,
    longitudes,
    dataset
):
    """
    Request elevation (m) for a batch of coordinates
    from one OpenTopoData dataset.

    Returns a list of elevations in the same order as
    the input coordinates. Voids are returned as None.
    """

    elevations = []


    for start in range(0, len(latitudes), BATCH_SIZE):

        batch_lat = latitudes[start:start + BATCH_SIZE]

        batch_lon = longitudes[start:start + BATCH_SIZE]


        locations = "|".join(
            f"{lat},{lon}"
            for lat, lon in zip(batch_lat, batch_lon)
        )


        params = {

            "locations": locations,

            "interpolation": "bilinear"
        }


        response = requests.get(
            f"{BASE_URL}/{dataset}",
            params=params,
            timeout=REQUEST_TIMEOUT
        )


        response.raise_for_status()


        results = response.json()["results"]


        elevations.extend(
            result["elevation"]
            for result in results
        )


        time.sleep(
            REQUEST_DELAY
        )


    return elevations


# ============================================================
# 5. EXTRACT PRIMARY DATASET (SRTM 30 m)
# ============================================================

print("\n")
print("=" * 65)
print("NASA SRTM ELEVATION EXTRACTION")
print("=" * 65)

print(
    f"Primary dataset:   {PRIMARY_DATASET}"
)

print(
    f"Fallback datasets: {', '.join(FALLBACK_DATASETS)}"
)

print(
    f"Grid cells:        {len(grid)}"
)

print("=" * 65)
print()


latitudes = grid["query_latitude"].tolist()

longitudes = grid["query_longitude"].tolist()


primary_elevations = get_elevations(
    latitudes=latitudes,
    longitudes=longitudes,
    dataset=PRIMARY_DATASET
)


df = grid[
    [
        "grid_number",
        "grid_id",
        "query_latitude",
        "query_longitude"
    ]
].copy()


df["elevation_m"] = pd.to_numeric(
    pd.Series(primary_elevations, index=df.index, dtype="object"),
    errors="coerce"
)

df["elevation_source"] = PRIMARY_DATASET

df.loc[df["elevation_m"].isna(), "elevation_source"] = None


print(
    f"Voids in {PRIMARY_DATASET}: "
    f"{df['elevation_m'].isna().sum()}"
)


# ============================================================
# 6. FILL VOIDS WITH FALLBACK DATASETS
# ============================================================

for dataset in FALLBACK_DATASETS:

    void_mask = df["elevation_m"].isna()

    if not void_mask.any():
        break


    fallback_elevations = pd.to_numeric(
        pd.Series(
            get_elevations(
                latitudes=df.loc[void_mask, "query_latitude"].tolist(),
                longitudes=df.loc[void_mask, "query_longitude"].tolist(),
                dataset=dataset
            ),
            index=df.index[void_mask],
            dtype="object"
        ),
        errors="coerce"
    )


    filled = fallback_elevations.dropna()

    df.loc[filled.index, "elevation_m"] = filled

    df.loc[filled.index, "elevation_source"] = dataset


    print(
        f"Filled from {dataset}: "
        f"{len(filled)} of {void_mask.sum()}"
    )


# ============================================================
# 7. DATA VALIDATION
# ============================================================

print("\n")
print("=" * 65)
print("DATA VALIDATION")
print("=" * 65)

print(
    f"Remaining missing elevations: "
    f"{df['elevation_m'].isna().sum()}"
)

print(
    f"Points at or below sea level: "
    f"{(df['elevation_m'] <= 0).sum()}"
)

print("\nElevation summary (m):")

print(
    df["elevation_m"].describe()
)


if df["elevation_m"].isna().any():

    print("\nWARNING: grid cells without elevation:")

    print(
        df[df["elevation_m"].isna()].to_string(
            index=False
        )
    )


# ============================================================
# 8. SAVE DATASET
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 65)
print("SRTM ELEVATION EXTRACTION COMPLETE")
print("=" * 65)

print(
    f"Grid cells:       "
    f"{len(df)}"
)

print(
    f"Source counts:\n"
    f"{df['elevation_source'].value_counts().to_string()}"
)

print(
    f"\nOutput file:\n"
    f"{OUTPUT_FILE}"
)


# ============================================================
# 10. DISPLAY SAMPLE
# ============================================================

print("\nFirst 10 records:")

print(
    df.head(10).to_string(
        index=False
    )
)


print("\n")
print("Done.")


Loading grid metadata...
Grid cells found: 93


NASA SRTM ELEVATION EXTRACTION
Primary dataset:   srtm30m
Fallback datasets: srtm90m, aster30m
Grid cells:        93

Voids in srtm30m: 1
Filled from srtm90m: 0 of 1
Filled from aster30m: 1 of 1


DATA VALIDATION
Remaining missing elevations: 0
Points at or below sea level: 0

Elevation summary (m):
count      93.000000
mean      237.225806
std       293.795815
min         4.000000
25%        41.000000
50%       124.000000
75%       355.000000
max      1493.000000
Name: elevation_m, dtype: float64


SRTM ELEVATION EXTRACTION COMPLETE
Grid cells:       93
Source counts:
elevation_source
srtm30m     92
aster30m     1

Output file:
data\raw\elevation\grid_elevation.csv

First 10 records:
 grid_number         grid_id  query_latitude  query_longitude  elevation_m elevation_source
           1 G_LAT-4_LON-119        4.890420       119.821795         12.0          srtm30m
           2 G_LAT-4_LON-120        4.976409       120.006294         15.